In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import StructType,StructField,StringType, IntegerType, FloatType 

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/16 09:54:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.option("header","true").csv("output/emp/*.csv")

26/06/16 09:54:24 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: output/emp/*.csv.
java.io.FileNotFoundException: File output/emp/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSo

In [4]:
df.show(30)

+-----------+-------------+-------------+---+------+-------+----------+
|employee_id|department_id|         name|age|gender| salary|hired_date|
+-----------+-------------+-------------+---+------+-------+----------+
|         17|          105|  George Wang| 34|  Male|57000.0|2016-03-15|
|         19|          103|  Steven Chen| 36|  Male|62000.0|2015-08-01|
|         20|          102|    Grace Kim| 32|Female|53000.0|2018-11-01|
|          7|          101|James Johnson| 42|  Male|70000.0|2012-03-15|
|          8|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|
|          9|          103|      Tom Tan| 33|  Male|58000.0|2016-06-01|
|         11|          104|   David Park| 38|  Male|65000.0|2015-11-01|
|         12|          105|   Susan Chen| 31|Female|54000.0|2017-02-15|
|          5|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|
|          6|          103|    Jill Wong| 32|Female|52000.0|2018-07-01|
|         15|          106|  Michael Lee| 37|  Male|63000.0|2014

In [5]:
emp_dpt = df.select("department_id").distinct()
emp_dpt.show()

+-------------+
|department_id|
+-------------+
|          102|
|          103|
|          105|
|          101|
|          104|
|          106|
+-------------+



In [6]:
df = df.withColumn (
    "max_salary",
    max("salary").over(Window.partitionBy(col("department_id")))
)

df.show()

+-----------+-------------+-------------+---+------+-------+----------+----------+
|employee_id|department_id|         name|age|gender| salary|hired_date|max_salary|
+-----------+-------------+-------------+---+------+-------+----------+----------+
|          7|          101|James Johnson| 42|  Male|70000.0|2012-03-15|   70000.0|
|         20|          102|    Grace Kim| 32|Female|53000.0|2018-11-01|   55000.0|
|          8|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|   55000.0|
|          3|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|   55000.0|
|         19|          103|  Steven Chen| 36|  Male|62000.0|2015-08-01|   62000.0|
|          9|          103|      Tom Tan| 33|  Male|58000.0|2016-06-01|   62000.0|
|          5|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|   62000.0|
|          6|          103|    Jill Wong| 32|Female|52000.0|2018-07-01|   62000.0|
|         11|          104|   David Park| 38|  Male|65000.0|2015-11-01|   65000.0|
|   

In [7]:
df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Window [max(salary#22) windowspecdefinition(department_id#18, specifiedwindowframe(RowFrame, unboundedpreceding$(), unboundedfollowing$())) AS max_salary#59], [department_id#18]
   +- Sort [department_id#18 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(department_id#18, 200), ENSURE_REQUIREMENTS, [plan_id=131]
         +- FileScan csv [employee_id#17,department_id#18,name#19,age#20,gender#21,salary#22,hired_date#23] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(8 paths)[file:/Users/AnhHuynh/Documents/PySpark/output/emp/part-00000-a2426edd-..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<employee_id:string,department_id:string,name:string,age:string,gender:string,salary:string...




In [8]:
# Using dense_rank to keep all the values with the same rank

# Find the 2nd highest salary

window_spec = Window.partitionBy("department_id").orderBy(col("salary").desc())
df_with_rank = df.withColumn("rank", dense_rank().over(window_spec))
second_highest = df_with_rank.filter(col("rank") == 2)
second_highest.show()


+-----------+-------------+-----------+---+------+-------+----------+----------+----+
|employee_id|department_id|       name|age|gender| salary|hired_date|max_salary|rank|
+-----------+-------------+-----------+---+------+-------+----------+----------+----+
|         20|          102|  Grace Kim| 32|Female|53000.0|2018-11-01|   55000.0|   2|
|          5|          103|  Jack Chan| 40|  Male|60000.0|2013-04-01|   62000.0|   2|
|         12|          105| Susan Chen| 31|Female|54000.0|2017-02-15|   57000.0|   2|
|         15|          106|Michael Lee| 37|  Male|63000.0|2014-09-30|   75000.0|   2|
+-----------+-------------+-----------+---+------+-------+----------+----------+----+



In [9]:
row_num = Window.partitionBy("department_id").orderBy(col("salary").desc())

df_row_num = df.withColumn("rn", row_number().over(row_num))

second_highest_1 = df_row_num.filter(col("rn")==1)

second_highest_1.show()

+-----------+-------------+-------------+---+------+-------+----------+----------+---+
|employee_id|department_id|         name|age|gender| salary|hired_date|max_salary| rn|
+-----------+-------------+-------------+---+------+-------+----------+----------+---+
|          7|          101|James Johnson| 42|  Male|70000.0|2012-03-15|   70000.0|  1|
|          3|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|   55000.0|  1|
|         19|          103|  Steven Chen| 36|  Male|62000.0|2015-08-01|   62000.0|  1|
|         11|          104|   David Park| 38|  Male|65000.0|2015-11-01|   65000.0|  1|
|         17|          105|  George Wang| 34|  Male|57000.0|2016-03-15|   57000.0|  1|
|         13|          106|    Brian Kim| 45|  Male|75000.0|2011-07-01|   75000.0|  1|
+-----------+-------------+-------------+---+------+-------+----------+----------+---+



In [10]:
df_row_num.show()

+-----------+-------------+-------------+---+------+-------+----------+----------+---+
|employee_id|department_id|         name|age|gender| salary|hired_date|max_salary| rn|
+-----------+-------------+-------------+---+------+-------+----------+----------+---+
|          7|          101|James Johnson| 42|  Male|70000.0|2012-03-15|   70000.0|  1|
|          3|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|   55000.0|  1|
|         20|          102|    Grace Kim| 32|Female|53000.0|2018-11-01|   55000.0|  2|
|          8|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|   55000.0|  3|
|         19|          103|  Steven Chen| 36|  Male|62000.0|2015-08-01|   62000.0|  1|
|          5|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|   62000.0|  2|
|          9|          103|      Tom Tan| 33|  Male|58000.0|2016-06-01|   62000.0|  3|
|          6|          103|    Jill Wong| 32|Female|52000.0|2018-07-01|   62000.0|  4|
|         11|          104|   David Park| 3

In [11]:
df_with_rank.show()

+-----------+-------------+-------------+---+------+-------+----------+----------+----+
|employee_id|department_id|         name|age|gender| salary|hired_date|max_salary|rank|
+-----------+-------------+-------------+---+------+-------+----------+----------+----+
|          7|          101|James Johnson| 42|  Male|70000.0|2012-03-15|   70000.0|   1|
|          3|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|   55000.0|   1|
|         20|          102|    Grace Kim| 32|Female|53000.0|2018-11-01|   55000.0|   2|
|          8|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|   55000.0|   3|
|         19|          103|  Steven Chen| 36|  Male|62000.0|2015-08-01|   62000.0|   1|
|          5|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|   62000.0|   2|
|          9|          103|      Tom Tan| 33|  Male|58000.0|2016-06-01|   62000.0|   3|
|          6|          103|    Jill Wong| 32|Female|52000.0|2018-07-01|   62000.0|   4|
|         11|          104|   Da

In [12]:
# using expr

emp3 = df.withColumn("rn_expr", expr("row_number() over(partition by department_id order by salary desc)")).where("rn_expr = 2")
emp3.show()

+-----------+-------------+-----------+---+------+-------+----------+----------+-------+
|employee_id|department_id|       name|age|gender| salary|hired_date|max_salary|rn_expr|
+-----------+-------------+-----------+---+------+-------+----------+----------+-------+
|         20|          102|  Grace Kim| 32|Female|53000.0|2018-11-01|   55000.0|      2|
|          5|          103|  Jack Chan| 40|  Male|60000.0|2013-04-01|   62000.0|      2|
|         12|          105| Susan Chen| 31|Female|54000.0|2017-02-15|   57000.0|      2|
|         15|          106|Michael Lee| 37|  Male|63000.0|2014-09-30|   75000.0|      2|
+-----------+-------------+-----------+---+------+-------+----------+----------+-------+



#### Repartition and Join

In [13]:
# Create a department dataframe
dept_schema = StructType([
    StructField("department_id",IntegerType(), False),
    StructField("department_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("country", StringType(), True),
    StructField("budget", FloatType(),True)
])


dept_data = [
    [101, "Sales", "NYC", "US", 1000000.0],
    [102, "Marketing", "LA", "US", 900000.0],
    [103, "Finance", "London", "UK", 1200000.0],
    [104, "Engineering", "Beijing", "China", 1500000.0],
    [105, "Human Resources", "Tokyo", "Japan", 800000.0],
    [106, "Research and Development", "Perth", "Australia", 1100000.0],
    [107, "Customer Service", "Sydney", "Australia", 950000.0]
]

dept = spark.createDataFrame(dept_data,schema=dept_schema)
dept.show()

+-------------+--------------------+-------+---------+---------+
|department_id|     department_name|   city|  country|   budget|
+-------------+--------------------+-------+---------+---------+
|          101|               Sales|    NYC|       US|1000000.0|
|          102|           Marketing|     LA|       US| 900000.0|
|          103|             Finance| London|       UK|1200000.0|
|          104|         Engineering|Beijing|    China|1500000.0|
|          105|     Human Resources|  Tokyo|    Japan| 800000.0|
|          106|Research and Deve...|  Perth|Australia|1100000.0|
|          107|    Customer Service| Sydney|Australia| 950000.0|
+-------------+--------------------+-------+---------+---------+



In [14]:
# Emp data and schema
emp_schema = StructType([
    StructField("employee_id",IntegerType(),False),
    StructField("department_id",IntegerType(), True),
    StructField("name", StringType(),False),
    StructField("age",IntegerType(),False),
    StructField("gender",StringType(),True),
    StructField("salary",FloatType(),False),
    StructField("hired_date",StringType(),False)
])
emp_data = [
    [1, 101, "John Doe", 30, "Male", 50000.0, "2015-01-01"],
    [2, 101, "Jane Smith", 25, "Female", 45000.0, "2016-02-15"],
    [3, 102, "Bob Brown", 35, "Male", 55000.0, "2014-05-01"],
    [4, 102, "Alice Lee", 28, "Female", 48000.0, "2017-09-30"],
    [5, 103, "Jack Chan", 40, "Male", 60000.0, "2013-04-01"],
    [6, 103, "Jill Wong", 32, "Female", 52000.0, "2018-07-01"],
    [7, 101, "James Johnson", 42, "Male", 70000.0, "2012-03-15"],
    [8, 102, "Kate Kim", 29, "Female", 51000.0, "2019-10-01"],
    [9, 103, "Tom Tan", 33, "Male", 58000.0, "2016-06-01"],
    [10, 104, "Lisa Lee", 27, "Female", 47000.0, "2018-08-01"],
    [11, 104, "David Park", 38, "Male", 65000.0, "2015-11-01"],
    [12, 105, "Susan Chen", 31, "Female", 54000.0, "2017-02-15"],
    [13, 106, "Brian Kim", 45, "Male", 75000.0, "2011-07-01"],
    [14, 107, "Emily Lee", 26, "Female", 46000.0, "2019-01-01"],
    [15, 106, "Michael Lee", 37, "Male", 63000.0, "2014-09-30"],
    [16, 107, "Kelly Zhang", 30, "Female", 49000.0, "2018-04-01"],
    [17, 105, "George Wang", 34, "Male", 57000.0, "2016-03-15"],
    [18, 104, "Nancy Liu", 29, "Female", 50000.0, "2017-06-01"],
    [19, 103, "Steven Chen", 36, "Male", 62000.0, "2015-08-01"],
    [20, 102, "Grace Kim", 32, "Female", 53000.0, "2018-11-01"]
]

employee = spark.createDataFrame(emp_data,schema=emp_schema)
employee.show()

+-----------+-------------+-------------+---+------+-------+----------+
|employee_id|department_id|         name|age|gender| salary|hired_date|
+-----------+-------------+-------------+---+------+-------+----------+
|          1|          101|     John Doe| 30|  Male|50000.0|2015-01-01|
|          2|          101|   Jane Smith| 25|Female|45000.0|2016-02-15|
|          3|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|
|          4|          102|    Alice Lee| 28|Female|48000.0|2017-09-30|
|          5|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|
|          6|          103|    Jill Wong| 32|Female|52000.0|2018-07-01|
|          7|          101|James Johnson| 42|  Male|70000.0|2012-03-15|
|          8|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|
|          9|          103|      Tom Tan| 33|  Male|58000.0|2016-06-01|
|         10|          104|     Lisa Lee| 27|Female|47000.0|2018-08-01|
|         11|          104|   David Park| 38|  Male|65000.0|2015

In [15]:
employee.printSchema()

root
 |-- employee_id: integer (nullable = false)
 |-- department_id: integer (nullable = true)
 |-- name: string (nullable = false)
 |-- age: integer (nullable = false)
 |-- gender: string (nullable = true)
 |-- salary: float (nullable = false)
 |-- hired_date: string (nullable = false)



In [16]:
employee.rdd.getNumPartitions()


8

In [17]:
dept.rdd.getNumPartitions()

8

In [18]:
employee_partition = employee.repartition(4,"department_id")

In [19]:
employee_1 = employee.withColumn("partion_number", spark_partition_id())
employee_1 = employee_1.repartition(4, "department_id").withColumn("partition_num", spark_partition_id())

employee_1.show()

+-----------+-------------+-------------+---+------+-------+----------+--------------+-------------+
|employee_id|department_id|         name|age|gender| salary|hired_date|partion_number|partition_num|
+-----------+-------------+-------------+---+------+-------+----------+--------------+-------------+
|          1|          101|     John Doe| 30|  Male|50000.0|2015-01-01|             0|            0|
|          2|          101|   Jane Smith| 25|Female|45000.0|2016-02-15|             0|            0|
|          7|          101|James Johnson| 42|  Male|70000.0|2012-03-15|             3|            0|
|         14|          107|    Emily Lee| 26|Female|46000.0|2019-01-01|             5|            0|
|         16|          107|  Kelly Zhang| 30|Female|49000.0|2018-04-01|             6|            0|
|         12|          105|   Susan Chen| 31|Female|54000.0|2017-02-15|             4|            1|
|         13|          106|    Brian Kim| 45|  Male|75000.0|2011-07-01|             5|     

In [20]:
e = employee.alias("e")
d = dept.alias("d")

df_joined = e.join(d,how="inner",on=e.department_id==d.department_id).drop(d.department_id)
data = df_joined.select(e.name, d.department_name, e.salary).show()

+-------------+--------------------+-------+
|         name|     department_name| salary|
+-------------+--------------------+-------+
|     John Doe|               Sales|50000.0|
|   Jane Smith|               Sales|45000.0|
|James Johnson|               Sales|70000.0|
|    Bob Brown|           Marketing|55000.0|
|    Alice Lee|           Marketing|48000.0|
|     Kate Kim|           Marketing|51000.0|
|    Grace Kim|           Marketing|53000.0|
|    Jack Chan|             Finance|60000.0|
|    Jill Wong|             Finance|52000.0|
|      Tom Tan|             Finance|58000.0|
|  Steven Chen|             Finance|62000.0|
|     Lisa Lee|         Engineering|47000.0|
|   David Park|         Engineering|65000.0|
|    Nancy Liu|         Engineering|50000.0|
|   Susan Chen|     Human Resources|54000.0|
|  George Wang|     Human Resources|57000.0|
|    Brian Kim|Research and Deve...|75000.0|
|  Michael Lee|Research and Deve...|63000.0|
|    Emily Lee|    Customer Service|46000.0|
|  Kelly Z

In [21]:
# Joining with cascading conditions

df_final = e.join(d,how="left_outer",
                  on=(e.department_id == d.department_id) & ((e.department_id == 101) | (e.department_id == 102))
)


In [22]:
df_final.show()

+-----------+-------------+-------------+---+------+-------+----------+-------------+---------------+----+-------+---------+
|employee_id|department_id|         name|age|gender| salary|hired_date|department_id|department_name|city|country|   budget|
+-----------+-------------+-------------+---+------+-------+----------+-------------+---------------+----+-------+---------+
|          1|          101|     John Doe| 30|  Male|50000.0|2015-01-01|          101|          Sales| NYC|     US|1000000.0|
|          2|          101|   Jane Smith| 25|Female|45000.0|2016-02-15|          101|          Sales| NYC|     US|1000000.0|
|          3|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|          102|      Marketing|  LA|     US| 900000.0|
|          4|          102|    Alice Lee| 28|Female|48000.0|2017-09-30|          102|      Marketing|  LA|     US| 900000.0|
|          5|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|         NULL|           NULL|NULL|   NULL|     NULL|


In [23]:
df_final.rdd

MapPartitionsRDD[139] at javaToPython at NativeMethodAccessorImpl.java:0

26/06/16 12:11:51 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 619232 ms exceeds timeout 120000 ms
26/06/16 12:11:51 WARN SparkContext: Killing executors is not supported by current scheduler.
26/06/16 12:11:52 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1363)
	at o